In [1]:

# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
#  2  Imports & Setup

import os
import copy
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import torchvision.models as models

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True   # speeds up training with fixed input size

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
# 3 Paths & Hyperparameters

BASE          = '/kaggle/input/competitions/plant-leaves-super-resolution-challenge'
LR_TRAIN_DIR  = f'{BASE}/train_Low_Resolution'
HR_TRAIN_DIR  = f'{BASE}/train_High_Resolution'
TEST_DIR      = f'{BASE}/test_Low_Resolution'
VGG_WEIGHTS   = f'{BASE}/vgg19_weights.pth'
WORKDIR       = '/kaggle/working'

# ── Model ──────────────────────────────
NUM_RRDB_BLOCKS  = 8       # RRDB blocks (heavier than simple res blocks)
NUM_CHANNELS     = 64
GROWTH_RATE      = 32      # dense layer growth rate inside each RDB

# ── Training ───────────────────────────
BATCH_SIZE       = 12      # slightly smaller to fit RRDB in VRAM
LR_G             = 1e-4
LR_D             = 1e-4
PHASE1_EPOCHS    = 60      # L1+SSIM pre-training
PHASE2_EPOCHS    = 60      # full GAN fine-tuning (increased from 40)

# Save a checkpoint every N epochs in Phase 2 (for Model Soup)
CKPT_EVERY       = 10

# ── Loss weights (Phase 2) ─────────────
W_PIXEL = 0.8      # L1 pixel
W_SSIM  = 0.2      # SSIM structural
W_PERC  = 0.1      # VGG perceptual
W_ADV   = 0.001    # adversarial — keep tiny!

print("Paths:")
for d in [LR_TRAIN_DIR, HR_TRAIN_DIR, TEST_DIR, VGG_WEIGHTS]:
    print(f"  {'OK' if os.path.exists(d) else 'MISSING'} — {d}")
print(f"\nTrain pairs : {len(os.listdir(LR_TRAIN_DIR))}")
print(f"Test images : {len(os.listdir(TEST_DIR))}")

Paths:
  OK — /kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution
  OK — /kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_High_Resolution
  OK — /kaggle/input/competitions/plant-leaves-super-resolution-challenge/test_Low_Resolution
  OK — /kaggle/input/competitions/plant-leaves-super-resolution-challenge/vgg19_weights.pth

Train pairs : 1642
Test images : 495


In [4]:
#  4  Dataset

class SRDataset(Dataset):
    """
    Paired LR (32x32) + HR (128x128) dataset with on-the-fly augmentation.
    In test mode (hr_dir=None) returns (lr_tensor, filename).
    """
    def __init__(self, lr_dir, hr_dir=None, augment=True):
        self.lr_dir  = lr_dir
        self.hr_dir  = hr_dir
        self.augment = augment
        self.files   = sorted(os.listdir(lr_dir))

    def __len__(self):
        return len(self.files)

    def _augment_pair(self, lr, hr):
        if random.random() > 0.5:
            lr, hr = TF.hflip(lr), TF.hflip(hr)
        if random.random() > 0.5:
            lr, hr = TF.vflip(lr), TF.vflip(hr)
        angle = random.choice([0, 90, 180, 270])
        if angle:
            lr, hr = TF.rotate(lr, angle), TF.rotate(hr, angle)
        return lr, hr

    def __getitem__(self, idx):
        fname     = self.files[idx]
        lr_tensor = TF.to_tensor(Image.open(os.path.join(self.lr_dir, fname)).convert('RGB'))
        if self.hr_dir is not None:
            hr_tensor = TF.to_tensor(Image.open(os.path.join(self.hr_dir, fname)).convert('RGB'))
            if self.augment:
                lr_tensor, hr_tensor = self._augment_pair(lr_tensor, hr_tensor)
            return lr_tensor, hr_tensor
        return lr_tensor, fname


# Sanity check
_ds = SRDataset(LR_TRAIN_DIR, HR_TRAIN_DIR)
_lr, _hr = _ds[0]
print(f"LR: {_lr.shape}  HR: {_hr.shape}")
print(f"LR range [{_lr.min():.2f}, {_lr.max():.2f}]  HR range [{_hr.min():.2f}, {_hr.max():.2f}]")
del _ds, _lr, _hr

LR: torch.Size([3, 32, 32])  HR: torch.Size([3, 128, 128])
LR range [0.00, 0.71]  HR range [0.00, 1.00]


In [5]:
#  5  Channel Attention


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 4)),
            nn.ReLU(inplace=True),
            nn.Linear(max(channels // reduction, 4), channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.avg_pool(x).view(b, c)   # (B, C)
        w = self.fc(w).view(b, c, 1, 1)   # (B, C, 1, 1)
        return x * w                       # channel-wise scaling

In [6]:
#  6 — RRDB Block

class DenseLayer(nn.Module):
    """One dense connection: concat input → conv → LeakyReLU."""
    def __init__(self, in_channels, growth_rate):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, growth_rate, kernel_size=3, padding=1)
        self.act  = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        out = self.act(self.conv(x))
        return torch.cat([x, out], dim=1)   # dense concat


class RDB(nn.Module):
    """Residual Dense Block: 5 dense layers → compress → channel attn → residual."""
    def __init__(self, channels, growth_rate):
        super().__init__()
        dense_layers = []
        in_ch = channels
        for _ in range(5):
            dense_layers.append(DenseLayer(in_ch, growth_rate))
            in_ch += growth_rate
        self.dense  = nn.Sequential(*dense_layers)
        # 1×1 conv to compress back to `channels`
        self.compress = nn.Conv2d(in_ch, channels, kernel_size=1)
        self.ca       = ChannelAttention(channels)
        self.scale    = 0.2   # residual scaling

    def forward(self, x):
        feat = self.dense(x)           # (B, channels+5*growth, H, W)
        feat = self.compress(feat)     # (B, channels, H, W)
        feat = self.ca(feat)           # channel attention
        return x + feat * self.scale   # residual


class RRDB(nn.Module):
    """Residual-in-Residual Dense Block: 3 stacked RDBs + outer residual."""
    def __init__(self, channels, growth_rate):
        super().__init__()
        self.rdb1  = RDB(channels, growth_rate)
        self.rdb2  = RDB(channels, growth_rate)
        self.rdb3  = RDB(channels, growth_rate)
        self.scale = 0.2

    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        return x + out * self.scale   # outer residual


# Quick shape test
_b  = RRDB(64, 32)
_in = torch.randn(1, 64, 32, 32)
print(f"RRDB output: {_b(_in).shape}")
del _b, _in

RRDB output: torch.Size([1, 64, 32, 32])


In [7]:
# 7  Generator
class Generator(nn.Module):
    def __init__(self, num_rrdb=8, channels=64, growth_rate=32):
        super().__init__()
        # 1. Feature extraction head
        self.head = nn.Sequential(
            nn.Conv2d(3, channels, kernel_size=9, padding=4),
            nn.LeakyReLU(0.2, inplace=True)
        )
        # 2. RRDB trunk
        self.trunk = nn.Sequential(*[RRDB(channels, growth_rate) for _ in range(num_rrdb)])
        # 3. Post-trunk conv (before global residual add)
        self.post_trunk = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        # 4. 4× upsampling — two PixelShuffle ×2 stages
        self.upsample = nn.Sequential(
            nn.Conv2d(channels, channels * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),        # ×2: 32→64
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),        # ×2: 64→128
            nn.LeakyReLU(0.2, inplace=True),
        )
        # 5. Output conv
        self.tail = nn.Sequential(
            nn.Conv2d(channels, 3, kernel_size=9, padding=4),
            nn.Sigmoid()               # [0, 1]
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        feat = self.head(x)                        # (B, C, 32, 32)
        trunk = self.post_trunk(self.trunk(feat))  # (B, C, 32, 32)
        feat  = feat + trunk                       # global residual
        feat  = self.upsample(feat)                # (B, C, 128, 128)
        return self.tail(feat)                     # (B, 3, 128, 128)


# Test
_g   = Generator(NUM_RRDB_BLOCKS, NUM_CHANNELS, GROWTH_RATE)
_in  = torch.randn(1, 3, 32, 32)
_out = _g(_in)
print(f"Generator output: {_out.shape}")
print(f"Generator params: {sum(p.numel() for p in _g.parameters()):,}")
del _g, _in, _out

Generator output: torch.Size([1, 3, 128, 128])
Generator params: 5,150,563


In [8]:
#  8  Discriminator (PatchGAN)

class DBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            DBlock(64,  64,  stride=2),
            DBlock(64,  128, stride=1),
            DBlock(128, 128, stride=2),
            DBlock(128, 256, stride=1),
            DBlock(256, 256, stride=2),
            DBlock(256, 512, stride=1),
            DBlock(512, 512, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 1),
            nn.Sigmoid()
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.normal_(m.weight, 0.0, 0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.classifier(self.features(x))


_d   = Discriminator()
_in  = torch.randn(1, 3, 128, 128)
print(f"Discriminator output: {_d(_in).shape}")
print(f"Discriminator params: {sum(p.numel() for p in _d.parameters()):,}")
del _d, _in

Discriminator output: torch.Size([1, 1])
Discriminator params: 5,215,425


In [9]:
# 9  VGG Perceptual Loss

class VGGPerceptualLoss(nn.Module):
    def __init__(self, vgg_weights_path):
        super().__init__()
        vgg = models.vgg19()
        vgg.load_state_dict(torch.load(vgg_weights_path, map_location='cpu'))
        self.extractor = nn.Sequential(*list(vgg.features)[:18]).eval()
        for p in self.extractor.parameters():
            p.requires_grad = False

    def forward(self, sr, hr):
        return nn.functional.l1_loss(self.extractor(sr), self.extractor(hr))


try:
    _v = VGGPerceptualLoss(VGG_WEIGHTS)
    print("VGG weights loaded OK")
    del _v
except Exception as e:
    print(f"VGG load error: {e}")

VGG weights loaded OK


In [10]:
#  10  Combined Pixel Loss (L1 + SSIM)

try:
    import subprocess
    subprocess.run(['pip', 'install', 'pytorch-msssim', '-q'], check=True)
    from pytorch_msssim import ssim as ssim_fn
    USE_SSIM = True
    print("pytorch-msssim loaded — using L1 + SSIM")
except Exception:
    USE_SSIM = False
    print("pytorch-msssim unavailable — using L1 only")

if USE_SSIM:
    def pixel_loss(sr, hr):
        l1    = nn.functional.l1_loss(sr, hr)
        _ssim = 1.0 - ssim_fn(sr, hr, data_range=1.0, size_average=True)
        return W_PIXEL * l1 + W_SSIM * _ssim
else:
    def pixel_loss(sr, hr):
        return nn.functional.l1_loss(sr, hr)

pytorch-msssim unavailable — using L1 only


ERROR: Could not find a version that satisfies the requirement pytorch-msssim (from versions: none)
ERROR: No matching distribution found for pytorch-msssim


In [11]:
# ─────────────────────────────────────────
# CELL 11 — Initialize Everything
# ─────────────────────────────────────────
train_dataset = SRDataset(LR_TRAIN_DIR, HR_TRAIN_DIR, augment=True)
train_loader  = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
print(f"Train batches per epoch: {len(train_loader)}")

generator     = Generator(NUM_RRDB_BLOCKS, NUM_CHANNELS, GROWTH_RATE).to(device)
discriminator = Discriminator().to(device)
vgg_loss_fn   = VGGPerceptualLoss(VGG_WEIGHTS).to(device)

criterion_BCE = nn.BCELoss()

print(f"Generator params    : {sum(p.numel() for p in generator.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in discriminator.parameters()):,}")

Train batches per epoch: 137
Generator params    : 5,150,563
Discriminator params: 5,215,425


In [12]:
#  12  Phase 1: Pre-Training with Pixel Loss Only

print("="*60)
print(f"Phase 1: Pixel-Loss Pre-Training ({PHASE1_EPOCHS} epochs)")
print("="*60)

opt_G1 = optim.Adam(generator.parameters(), lr=LR_G, betas=(0.9, 0.999))
scheduler_G1 = optim.lr_scheduler.CosineAnnealingLR(
    opt_G1, T_max=PHASE1_EPOCHS, eta_min=1e-6
)

best_loss_p1 = float('inf')

generator.train()
for epoch in range(1, PHASE1_EPOCHS + 1):
    running_loss = 0.0
    for lr_imgs, hr_imgs in train_loader:
        lr_imgs = lr_imgs.to(device)
        hr_imgs = hr_imgs.to(device)

        sr_imgs = generator(lr_imgs)
        loss    = pixel_loss(sr_imgs, hr_imgs)

        opt_G1.zero_grad()
        loss.backward()
        opt_G1.step()
        running_loss += loss.item()

    scheduler_G1.step()
    avg = running_loss / len(train_loader)

    if avg < best_loss_p1:
        best_loss_p1 = avg
        torch.save(generator.state_dict(), f'{WORKDIR}/gen_best_p1.pth')

    if epoch % 10 == 0:
        print(f"  Epoch {epoch:3d}/{PHASE1_EPOCHS}  |  Loss: {avg:.5f}  |  LR: {scheduler_G1.get_last_lr()[0]:.2e}")

torch.save(generator.state_dict(), f'{WORKDIR}/gen_final_p1.pth')
print(f"\nPhase 1 done. Best loss: {best_loss_p1:.5f}")

Phase 1: Pixel-Loss Pre-Training (60 epochs)
  Epoch  10/60  |  Loss: 0.07040  |  LR: 9.34e-05
  Epoch  20/60  |  Loss: 0.06938  |  LR: 7.52e-05
  Epoch  30/60  |  Loss: 0.06886  |  LR: 5.05e-05
  Epoch  40/60  |  Loss: 0.06855  |  LR: 2.58e-05
  Epoch  50/60  |  Loss: 0.06835  |  LR: 7.63e-06
  Epoch  60/60  |  Loss: 0.06829  |  LR: 1.00e-06

Phase 1 done. Best loss: 0.06829


In [13]:
#  13  Phase 2: Full GAN Fine-Tuning


print("="*60)
print(f"Phase 2: Full GAN Training ({PHASE2_EPOCHS} epochs)")
print("="*60)

generator.load_state_dict(torch.load(f'{WORKDIR}/gen_best_p1.pth'))

opt_G2 = optim.Adam(generator.parameters(),     lr=LR_G / 10,  betas=(0.9, 0.999))
opt_D2 = optim.Adam(discriminator.parameters(), lr=LR_D / 10,  betas=(0.9, 0.999))

scheduler_G2 = optim.lr_scheduler.CosineAnnealingLR(opt_G2, T_max=PHASE2_EPOCHS, eta_min=1e-7)
scheduler_D2 = optim.lr_scheduler.CosineAnnealingLR(opt_D2, T_max=PHASE2_EPOCHS, eta_min=1e-7)

best_g_loss = float('inf')
saved_ckpts = []   # track checkpoint paths for Model Soup

generator.train()
discriminator.train()

for epoch in range(1, PHASE2_EPOCHS + 1):
    g_running = 0.0
    d_running = 0.0

    for lr_imgs, hr_imgs in train_loader:
        lr_imgs = lr_imgs.to(device)
        hr_imgs = hr_imgs.to(device)
        bs      = lr_imgs.size(0)

        real_labels = torch.ones(bs, 1, device=device)
        fake_labels = torch.zeros(bs, 1, device=device)

        # ── Discriminator step ──────────────────────
        with torch.no_grad():
            sr_d = generator(lr_imgs)
        d_loss = 0.5 * (criterion_BCE(discriminator(hr_imgs), real_labels) +
                        criterion_BCE(discriminator(sr_d),    fake_labels))
        opt_D2.zero_grad()
        d_loss.backward()
        opt_D2.step()

        # ── Generator step ───────────────────────────
        sr_imgs  = generator(lr_imgs)
        d_score  = discriminator(sr_imgs)

        loss_px  = pixel_loss(sr_imgs, hr_imgs)
        loss_vgg = vgg_loss_fn(sr_imgs, hr_imgs)
        loss_adv = criterion_BCE(d_score, real_labels)

        g_loss = loss_px + W_PERC * loss_vgg + W_ADV * loss_adv

        opt_G2.zero_grad()
        g_loss.backward()
        opt_G2.step()

        g_running += g_loss.item()
        d_running += d_loss.item()

    scheduler_G2.step()
    scheduler_D2.step()

    avg_g = g_running / len(train_loader)
    avg_d = d_running / len(train_loader)

    # ── Save best checkpoint ─────────────────────
    if avg_g < best_g_loss:
        best_g_loss = avg_g
        torch.save(generator.state_dict(), f'{WORKDIR}/gen_best_p2.pth')

    # ── Save periodic checkpoints for Model Soup ─
    if epoch % CKPT_EVERY == 0:
        ckpt_path = f'{WORKDIR}/gen_ckpt_epoch{epoch}.pth'
        torch.save(generator.state_dict(), ckpt_path)
        saved_ckpts.append(ckpt_path)
        print(f"  [ckpt saved] epoch {epoch}")

    if epoch % 10 == 0:
        print(f"  Epoch {epoch:3d}/{PHASE2_EPOCHS}  |  G: {avg_g:.5f}  |  D: {avg_d:.5f}")

torch.save(generator.state_dict(), f'{WORKDIR}/gen_final_p2.pth')
print(f"\nPhase 2 done. Best G loss: {best_g_loss:.5f}")
print(f"Checkpoints saved: {saved_ckpts}")

Phase 2: Full GAN Training (60 epochs)
  [ckpt saved] epoch 10
  Epoch  10/60  |  G: 0.12557  |  D: 0.01150
  [ckpt saved] epoch 20
  Epoch  20/60  |  G: 0.12669  |  D: 0.00309
  [ckpt saved] epoch 30
  Epoch  30/60  |  G: 0.12727  |  D: 0.00129
  [ckpt saved] epoch 40
  Epoch  40/60  |  G: 0.12761  |  D: 0.00068
  [ckpt saved] epoch 50
  Epoch  50/60  |  G: 0.12781  |  D: 0.00051
  [ckpt saved] epoch 60
  Epoch  60/60  |  G: 0.12786  |  D: 0.00065

Phase 2 done. Best G loss: 0.12335
Checkpoints saved: ['/kaggle/working/gen_ckpt_epoch10.pth', '/kaggle/working/gen_ckpt_epoch20.pth', '/kaggle/working/gen_ckpt_epoch30.pth', '/kaggle/working/gen_ckpt_epoch40.pth', '/kaggle/working/gen_ckpt_epoch50.pth', '/kaggle/working/gen_ckpt_epoch60.pth']


In [14]:
#  14 Model Soup: Average Last N Checkpoints

def model_soup(checkpoint_paths):
    """
    Average the state_dicts from multiple checkpoints.
    Returns the averaged state_dict.
    """
    print(f"Averaging {len(checkpoint_paths)} checkpoints:")
    for p in checkpoint_paths:
        print(f"  {p}")

    averaged = copy.deepcopy(torch.load(checkpoint_paths[0], map_location='cpu'))
    for path in checkpoint_paths[1:]:
        state = torch.load(path, map_location='cpu')
        for key in averaged:
            averaged[key] = averaged[key] + state[key]
    for key in averaged:
        averaged[key] = averaged[key] / len(checkpoint_paths)
    return averaged


# Use the last 5 checkpoints (or however many were saved)
soup_ckpts = saved_ckpts[-5:] if len(saved_ckpts) >= 5 else saved_ckpts

if len(soup_ckpts) > 1:
    soups_state = model_soup(soup_ckpts)
    generator.load_state_dict(soups_state)
    print("\nModel Soup applied.")
else:
    # If only one checkpoint available, fall back to best
    generator.load_state_dict(torch.load(f'{WORKDIR}/gen_best_p2.pth'))
    print("Only one checkpoint — using best Phase 2 weights.")

generator.eval()
print("Generator set to eval mode.")

Averaging 5 checkpoints:
  /kaggle/working/gen_ckpt_epoch20.pth
  /kaggle/working/gen_ckpt_epoch30.pth
  /kaggle/working/gen_ckpt_epoch40.pth
  /kaggle/working/gen_ckpt_epoch50.pth
  /kaggle/working/gen_ckpt_epoch60.pth

Model Soup applied.
Generator set to eval mode.


In [15]:

#  15 Test-Time Augmentation (TTA)

def tta_predict(model, lr_tensor, device):
    """
    lr_tensor : (3, 32, 32) float in [0, 1]
    Returns   : (3, 128, 128) float, averaged over 8 augmentations
    """
    preds = []
    for hflip in [False, True]:
        for angle in [0, 90, 180, 270]:
            aug = lr_tensor.clone()
            if hflip:
                aug = TF.hflip(aug)
            if angle:
                aug = TF.rotate(aug, angle)

            with torch.no_grad():
                pred = model(aug.unsqueeze(0).to(device)).squeeze(0)  # (3,128,128)

            # Reverse augmentation on prediction
            if angle:
                pred = TF.rotate(pred, -angle)
            if hflip:
                pred = TF.hflip(pred)

            preds.append(pred)

    return torch.stack(preds).mean(0)   # (3, 128, 128)


print("TTA function ready (8 augmentations per image).")

TTA function ready (8 augmentations per image).


In [16]:

#  16  submission.csv

test_files = sorted(os.listdir(TEST_DIR))
print(f"Generating predictions for {len(test_files)} test images...")

rows = []
for i, fname in enumerate(test_files):
    lr_img    = Image.open(os.path.join(TEST_DIR, fname)).convert('RGB')
    lr_tensor = TF.to_tensor(lr_img)   # (3, 32, 32)

    sr_tensor = tta_predict(generator, lr_tensor, device)  # (3, 128, 128)

    # Denormalize: clamp → ×255 → round → uint8
    sr_np  = (sr_tensor.clamp(0, 1) * 255).round().byte().cpu().numpy()  # (3,128,128)
    sr_hwc = sr_np.transpose(1, 2, 0)   # (128, 128, 3) HWC
    flat   = sr_hwc.flatten()           # (49152,)

    rows.append({'Id': fname, 'Pixels': ' '.join(map(str, flat.tolist()))})

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(test_files)} done")

submission = pd.DataFrame(rows)
submission.to_csv(f'{WORKDIR}/submission.csv', index=False)
print(f"\nsubmission.csv saved — {submission.shape}")

# ── Validation ─────────────────────────────────────
print("\nValidation checks:")
sample_pixels = submission['Pixels'].iloc[0].split()
print(f"  Pixel count row 0 : {len(sample_pixels)}  (expected 49152)")
vals = [int(x) for x in sample_pixels]
print(f"  Pixel range row 0 : [{min(vals)}, {max(vals)}]  (expected [0, 255])")
print(f"  Sample filename   : {submission['Id'].iloc[0]}")
print("\nAll checks passed!")

Generating predictions for 495 test images...
  50/495 done
  100/495 done
  150/495 done
  200/495 done
  250/495 done
  300/495 done
  350/495 done
  400/495 done
  450/495 done

submission.csv saved — (495, 2)

Validation checks:
  Pixel count row 0 : 49152  (expected 49152)
  Pixel range row 0 : [4, 176]  (expected [0, 255])
  Sample filename   : agrivision_test_0000.png

All checks passed!
